# Week 3: Expanding Dataset to 200+ Images with Variety

This notebook expands the Week 1 dataset by generating additional
synthetic Urdu images with variety in fonts (Nastaliq, Naskh),
backgrounds (white, newspaper-yellow, dark signboard), and text sizes
(small to large), then preprocesses them for the OCR pipeline.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Download free Urdu fonts from Google Fonts
!wget -q https://github.com/google/fonts/raw/main/ofl/notonastaliqurdu/NotoNastaliqUrdu-Regular.ttf -O nastaliq.ttf
!wget -q https://github.com/google/fonts/raw/main/ofl/notonaskharabic/NotoNaskhArabic-Regular.ttf -O naskh.ttf

import os
print('Nastaliq downloaded:', os.path.exists('nastaliq.ttf'))
print('Naskh downloaded:', os.path.exists('naskh.ttf'))

Nastaliq downloaded: True
Naskh downloaded: True


In [4]:
!pip install arabic_reshaper python-bidi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 10.2 MB/s eta 0:00:00


In [6]:
!wget -q --show-progress https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoNastaliqUrdu/NotoNastaliqUrdu-Regular.ttf -O nastaliq.ttf
!wget -q --show-progress https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoNaskhArabic/NotoNaskhArabic-Regular.ttf -O naskh.ttf

import os

# Check file sizes - a real font file should be several hundred KB, not 0 bytes
nastaliq_size = os.path.getsize('nastaliq.ttf') if os.path.exists('nastaliq.ttf') else 0
naskh_size = os.path.getsize('naskh.ttf') if os.path.exists('naskh.ttf') else 0

print(f'nastaliq.ttf size: {nastaliq_size} bytes')
print(f'naskh.ttf size: {naskh_size} bytes')

if nastaliq_size < 1000 or naskh_size < 1000:
    print('WARNING: One or both fonts failed to download properly!')
else:
    print('Both fonts downloaded successfully!')

nastaliq.ttf        100%[===================>]   1.12M  --.-KB/s    in 0.04s   
naskh.ttf           100%[===================>] 174.21K  --.-KB/s    in 0.01s   
nastaliq.ttf size: 1171972 bytes
naskh.ttf size: 178388 bytes
Both fonts downloaded successfully!


In [7]:
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
import os
import random

# 20 new Urdu sentences
new_urdu_texts = [
    "پاکستان کا دارالحکومت اسلام آباد ہے",
    "چاند رات کو بہت خوبصورت لگتا ہے",
    "ہر انسان کو محنت کرنی چاہیے",
    "کتابیں علم کا خزانہ ہیں",
    "بارش کے بعد ہوا تازہ ہو جاتی ہے",
    "طالب علم کو وقت کی قدر کرنی چاہیے",
    "دوستی میں اعتماد بہت ضروری ہے",
    "صبح جلدی اٹھنا صحت کے لیے اچھا ہے",
    "ماں باپ کی خدمت بہت بڑی نیکی ہے",
    "ایمانداری بہترین پالیسی ہے",
    "غریبوں کی مدد کرنی چاہیے",
    "علم حاصل کرنے کی کوئی عمر نہیں",
    "پھول خوشبو اور خوبصورتی کی علامت ہیں",
    "پرندے آزادی کی علامت ہیں",
    "زندگی میں صبر کا پھل میٹھا ہوتا ہے",
    "نیک اعمال انسان کو کامیاب بناتے ہیں",
    "تعلیم ہر انسان کا بنیادی حق ہے",
    "محنتی انسان کبھی ناکام نہیں ہوتا",
    "وقت پر کام کرنا کامیابی کی کنجی ہے",
    "دوسروں کی عزت کرنی چاہیے"
]

# Backgrounds for variety
backgrounds = [
    {'color': 'white', 'size': (500, 100)},
    {'color': '#FFF9C4', 'size': (500, 100)},
    {'color': '#2B2B2B', 'size': (500, 100)},
]

# Fonts for variety
fonts_list = [
    {'path': 'nastaliq.ttf', 'name': 'nastaliq'},
    {'path': 'naskh.ttf', 'name': 'naskh'},
]

# Font sizes for variety
font_sizes = [24, 36, 48]

os.makedirs('/content/drive/MyDrive/data/raw/variety', exist_ok=True)

count = 0
for i, text in enumerate(new_urdu_texts):
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)

    bg = random.choice(backgrounds)
    font_choice = random.choice(fonts_list)
    size = random.choice(font_sizes)

    img = Image.new('RGB', bg['size'], color=bg['color'])
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font_choice['path'], size)

    text_color = 'black' if bg['color'] != '#2B2B2B' else 'white'
    draw.text((10, 20), bidi_text, font=font, fill=text_color)

    filename = f"variety_{i+1}_{font_choice['name']}_{size}.png"
    save_path = f"/content/drive/MyDrive/data/raw/variety/{filename}"
    img.save(save_path)
    count += 1
    print(f'Generated: {filename} -- {text}')

print(f'Done! Generated {count} new variety images')

Generated: variety_1_naskh_24.png -- پاکستان کا دارالحکومت اسلام آباد ہے
Generated: variety_2_naskh_24.png -- چاند رات کو بہت خوبصورت لگتا ہے
Generated: variety_3_naskh_24.png -- ہر انسان کو محنت کرنی چاہیے
Generated: variety_4_nastaliq_48.png -- کتابیں علم کا خزانہ ہیں
Generated: variety_5_nastaliq_24.png -- بارش کے بعد ہوا تازہ ہو جاتی ہے
Generated: variety_6_nastaliq_36.png -- طالب علم کو وقت کی قدر کرنی چاہیے
Generated: variety_7_naskh_48.png -- دوستی میں اعتماد بہت ضروری ہے
Generated: variety_8_naskh_24.png -- صبح جلدی اٹھنا صحت کے لیے اچھا ہے
Generated: variety_9_nastaliq_48.png -- ماں باپ کی خدمت بہت بڑی نیکی ہے
Generated: variety_10_naskh_48.png -- ایمانداری بہترین پالیسی ہے
Generated: variety_11_naskh_24.png -- غریبوں کی مدد کرنی چاہیے
Generated: variety_12_nastaliq_36.png -- علم حاصل کرنے کی کوئی عمر نہیں
Generated: variety_13_nastaliq_36.png -- پھول خوشبو اور خوبصورتی کی علامت ہیں
Generated: variety_14_naskh_48.png -- پرندے آزادی کی علامت ہیں
Generated: variety_15_naskh_48.p

In [8]:
import glob

# Count everything
all_current = glob.glob('repo_data/data/raw/**/*.png', recursive=True)
all_current += glob.glob('repo_data/data/raw/**/*.jpg', recursive=True)
new_variety = glob.glob('/content/drive/MyDrive/data/raw/variety/*.png')

print(f'Original images (from Week 1 GitHub repo): {len(all_current)}')
print(f'New variety images: {len(new_variety)}')
print(f'Total: {len(all_current) + len(new_variety)}')

Original images (from Week 1 GitHub repo): 0
New variety images: 20
Total: 20


In [9]:
import glob

# Count everything: original 106 + new variety images
all_current = glob.glob('repo_data/data/raw/**/*.png', recursive=True)
all_current += glob.glob('repo_data/data/raw/**/*.jpg', recursive=True)
new_variety = glob.glob('/content/drive/MyDrive/data/raw/variety/*.png')

print(f'Original images: {len(all_current)}')
print(f'New variety images: {len(new_variety)}')
print(f'Total: {len(all_current) + len(new_variety)}')

Original images: 0
New variety images: 20
Total: 20


In [10]:
!git clone https://github.com/fatimasajid31/urdu-ocr-codesaviours-si26-fatima.git repo_data

Cloning into 'repo_data'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (352/352), done.
remote: Compressing objects: 100% (346/346), done.
remote: Total 352 (delta 56), reused 140 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (352/352), 1.92 MiB | 6.25 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [11]:
import glob

all_current = glob.glob('repo_data/data/raw/**/*.png', recursive=True)
all_current += glob.glob('repo_data/data/raw/**/*.jpg', recursive=True)
new_variety = glob.glob('/content/drive/MyDrive/data/raw/variety/*.png')

print(f'Original images (from Week 1/2 GitHub repo): {len(all_current)}')
print(f'New variety images: {len(new_variety)}')
print(f'Total: {len(all_current) + len(new_variety)}')

Original images (from Week 1/2 GitHub repo): 106
New variety images: 20
Total: 126


In [12]:
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
import os
import random

# 40 more new Urdu sentences (batch 2)
batch2_texts = [
    "علم بغیر عمل کے بیکار ہے",
    "سچ بولنا ہمیشہ فائدہ مند ہوتا ہے",
    "ہمسائیوں کے ساتھ اچھا سلوک کریں",
    "درخت لگانا ماحول کے لیے مفید ہے",
    "پانی زندگی کے لیے بہت ضروری ہے",
    "بچوں کی تربیت والدین کی ذمہ داری ہے",
    "کامیابی کے لیے مستقل مزاجی ضروری ہے",
    "اچھی عادتیں انسان کو بہتر بناتی ہیں",
    "دنیا میں امن بہت ضروری ہے",
    "صحت مند رہنے کے لیے ورزش کریں",
    "کتاب انسان کی بہترین دوست ہے",
    "وقت کبھی واپس نہیں آتا",
    "نیکی کبھی ضائع نہیں ہوتی",
    "علم کی روشنی سے جہالت دور ہوتی ہے",
    "محنت کے بغیر کامیابی ممکن نہیں",
    "غصہ انسان کی سب سے بڑی کمزوری ہے",
    "دوسروں کی مدد کرنا نیکی ہے",
    "استاد قوم کا معمار ہوتا ہے",
    "زندگی مشکلات کا نام ہے",
    "امید پر دنیا قائم ہے",
    "خواب دیکھنے سے کچھ نہیں ہوتا، عمل کرنا ضروری ہے",
    "سچی محنت رنگ لاتی ہے",
    "علم کے بغیر انسان ادھورا ہے",
    "قوم کی ترقی تعلیم میں ہے",
    "اچھے دوست زندگی کا سرمایہ ہیں",
    "ایمانداری کامیابی کی کنجی ہے",
    "بڑوں کا احترام کرنا چاہیے",
    "غریبی میں بھی خوداری ضروری ہے",
    "زندگی میں صبر کا پھل میٹھا ہے",
    "علم انسان کو باشعور بناتا ہے",
    "دیانتداری بہترین خوبی ہے",
    "قناعت سب سے بڑی دولت ہے",
    "دوسروں پر اعتماد کرنا سیکھیں",
    "ہر کام میں برکت نیت پر ہے",
    "زندگی ایک سفر ہے، منزل نہیں",
    "علم عمل کے بغیر بے سود ہے",
    "قوم کی تعمیر نوجوانوں کے ہاتھ میں ہے",
    "سچائی کا راستہ مشکل مگر روشن ہے",
    "کامیابی کے پیچھے قربانی ہوتی ہے",
    "زندگی میں مقصد رکھنا ضروری ہے"
]

backgrounds = [
    {'color': 'white', 'size': (500, 100)},
    {'color': '#FFF9C4', 'size': (500, 100)},
    {'color': '#2B2B2B', 'size': (500, 100)},
]

fonts_list = [
    {'path': 'nastaliq.ttf', 'name': 'nastaliq'},
    {'path': 'naskh.ttf', 'name': 'naskh'},
]

font_sizes = [24, 36, 48]

os.makedirs('/content/drive/MyDrive/data/raw/variety', exist_ok=True)

count = 0
for i, text in enumerate(batch2_texts):
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)

    bg = random.choice(backgrounds)
    font_choice = random.choice(fonts_list)
    size = random.choice(font_sizes)

    img = Image.new('RGB', bg['size'], color=bg['color'])
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font_choice['path'], size)

    text_color = 'black' if bg['color'] != '#2B2B2B' else 'white'
    draw.text((10, 20), bidi_text, font=font, fill=text_color)

    filename = f"variety2_{i+1}_{font_choice['name']}_{size}.png"
    save_path = f"/content/drive/MyDrive/data/raw/variety/{filename}"
    img.save(save_path)
    count += 1
    print(f'Generated: {filename} -- {text}')

print(f'Done! Generated {count} more variety images (batch 2)')

Generated: variety2_1_naskh_24.png -- علم بغیر عمل کے بیکار ہے
Generated: variety2_2_nastaliq_24.png -- سچ بولنا ہمیشہ فائدہ مند ہوتا ہے
Generated: variety2_3_nastaliq_24.png -- ہمسائیوں کے ساتھ اچھا سلوک کریں
Generated: variety2_4_nastaliq_48.png -- درخت لگانا ماحول کے لیے مفید ہے
Generated: variety2_5_naskh_48.png -- پانی زندگی کے لیے بہت ضروری ہے
Generated: variety2_6_naskh_24.png -- بچوں کی تربیت والدین کی ذمہ داری ہے
Generated: variety2_7_nastaliq_24.png -- کامیابی کے لیے مستقل مزاجی ضروری ہے
Generated: variety2_8_nastaliq_36.png -- اچھی عادتیں انسان کو بہتر بناتی ہیں
Generated: variety2_9_nastaliq_48.png -- دنیا میں امن بہت ضروری ہے
Generated: variety2_10_nastaliq_48.png -- صحت مند رہنے کے لیے ورزش کریں
Generated: variety2_11_nastaliq_48.png -- کتاب انسان کی بہترین دوست ہے
Generated: variety2_12_nastaliq_36.png -- وقت کبھی واپس نہیں آتا
Generated: variety2_13_nastaliq_48.png -- نیکی کبھی ضائع نہیں ہوتی
Generated: variety2_14_naskh_48.png -- علم کی روشنی سے جہالت دور ہوتی ہے
Gener

In [13]:
import glob

all_current = glob.glob('repo_data/data/raw/**/*.png', recursive=True)
all_current += glob.glob('repo_data/data/raw/**/*.jpg', recursive=True)
new_variety = glob.glob('/content/drive/MyDrive/data/raw/variety/*.png')

print(f'Original images (from Week 1/2 GitHub repo): {len(all_current)}')
print(f'New variety images: {len(new_variety)}')
print(f'Total: {len(all_current) + len(new_variety)}')

Original images (from Week 1/2 GitHub repo): 106
New variety images: 60
Total: 166


In [14]:
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
import os
import random

# 40 more new Urdu sentences (batch 3)
batch3_texts = [
    "اچھے اخلاق انسان کی پہچان ہیں",
    "علم کے بغیر ترقی ممکن نہیں",
    "محنت کبھی رائیگاں نہیں جاتی",
    "زندگی میں ہمت نہیں ہارنی چاہیے",
    "دوسروں کے کام آنا بڑی نیکی ہے",
    "وقت کی پابندی کامیابی کی نشانی ہے",
    "بچوں کو اچھی تربیت دینی چاہیے",
    "صبر اور استقامت سے سب کچھ ممکن ہے",
    "غرور انسان کو تباہ کر دیتا ہے",
    "علم روشنی ہے اور جہالت اندھیرا",
    "اپنے وطن سے محبت کرنی چاہیے",
    "سچائی اور ایمانداری کامیابی کی ضمانت ہے",
    "زندگی میں نظم و ضبط ضروری ہے",
    "استاد کا احترام کرنا فرض ہے",
    "علم انسان کی معراج ہے",
    "قوموں کی ترقی محنت سے ہوتی ہے",
    "دوسروں کے جذبات کا خیال رکھیں",
    "کامیاب لوگ ہمیشہ محنتی ہوتے ہیں",
    "زندگی میں مثبت سوچ رکھیں",
    "دیانتداری سب سے بڑی خوبی ہے",
    "علم حاصل کرنا فرض ہے",
    "دوستی میں خلوص ضروری ہے",
    "ماحول کی حفاظت ہماری ذمہ داری ہے",
    "بڑوں کی بات سننی چاہیے",
    "زندگی میں شکر گزار رہنا چاہیے",
    "اچھے کام کا صلہ ضرور ملتا ہے",
    "علم سے انسان کی سوچ بدلتی ہے",
    "دوسروں کی مدد سے دل کو سکون ملتا ہے",
    "زندگی میں توازن رکھنا ضروری ہے",
    "سچی خوشی دوسروں کو خوش دیکھ کر ملتی ہے",
    "علم کی تلاش کبھی ختم نہیں ہوتی",
    "قوم کی خدمت سب سے بڑی عبادت ہے",
    "اچھی صحبت انسان کو بہتر بناتی ہے",
    "زندگی میں ایمانداری کا دامن نہ چھوڑیں",
    "علم و عمل کا امتزاج کامیابی ہے",
    "دوسروں کی عزت کرنے سے عزت ملتی ہے",
    "زندگی میں مقصد کا تعین ضروری ہے",
    "سچی محنت ہمیشہ رنگ لاتی ہے",
    "علم انسان کو باوقار بناتا ہے",
    "قناعت پسندی زندگی کو آسان بناتی ہے"
]

backgrounds = [
    {'color': 'white', 'size': (500, 100)},
    {'color': '#FFF9C4', 'size': (500, 100)},
    {'color': '#2B2B2B', 'size': (500, 100)},
]

fonts_list = [
    {'path': 'nastaliq.ttf', 'name': 'nastaliq'},
    {'path': 'naskh.ttf', 'name': 'naskh'},
]

font_sizes = [24, 36, 48]

os.makedirs('/content/drive/MyDrive/data/raw/variety', exist_ok=True)

count = 0
for i, text in enumerate(batch3_texts):
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)

    bg = random.choice(backgrounds)
    font_choice = random.choice(fonts_list)
    size = random.choice(font_sizes)

    img = Image.new('RGB', bg['size'], color=bg['color'])
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font_choice['path'], size)

    text_color = 'black' if bg['color'] != '#2B2B2B' else 'white'
    draw.text((10, 20), bidi_text, font=font, fill=text_color)

    filename = f"variety3_{i+1}_{font_choice['name']}_{size}.png"
    save_path = f"/content/drive/MyDrive/data/raw/variety/{filename}"
    img.save(save_path)
    count += 1
    print(f'Generated: {filename} -- {text}')

print(f'Done! Generated {count} more variety images (batch 3)')

Generated: variety3_1_nastaliq_36.png -- اچھے اخلاق انسان کی پہچان ہیں
Generated: variety3_2_nastaliq_36.png -- علم کے بغیر ترقی ممکن نہیں
Generated: variety3_3_nastaliq_24.png -- محنت کبھی رائیگاں نہیں جاتی
Generated: variety3_4_naskh_36.png -- زندگی میں ہمت نہیں ہارنی چاہیے
Generated: variety3_5_nastaliq_36.png -- دوسروں کے کام آنا بڑی نیکی ہے
Generated: variety3_6_nastaliq_48.png -- وقت کی پابندی کامیابی کی نشانی ہے
Generated: variety3_7_nastaliq_48.png -- بچوں کو اچھی تربیت دینی چاہیے
Generated: variety3_8_naskh_24.png -- صبر اور استقامت سے سب کچھ ممکن ہے
Generated: variety3_9_nastaliq_48.png -- غرور انسان کو تباہ کر دیتا ہے
Generated: variety3_10_naskh_36.png -- علم روشنی ہے اور جہالت اندھیرا
Generated: variety3_11_nastaliq_36.png -- اپنے وطن سے محبت کرنی چاہیے
Generated: variety3_12_naskh_24.png -- سچائی اور ایمانداری کامیابی کی ضمانت ہے
Generated: variety3_13_nastaliq_36.png -- زندگی میں نظم و ضبط ضروری ہے
Generated: variety3_14_naskh_36.png -- استاد کا احترام کرنا فرض ہے
Genera

In [15]:
import glob

all_current = glob.glob('repo_data/data/raw/**/*.png', recursive=True)
all_current += glob.glob('repo_data/data/raw/**/*.jpg', recursive=True)
new_variety = glob.glob('/content/drive/MyDrive/data/raw/variety/*.png')

print(f'Original images (from Week 1/2 GitHub repo): {len(all_current)}')
print(f'New variety images: {len(new_variety)}')
print(f'Total: {len(all_current) + len(new_variety)}')

Original images (from Week 1/2 GitHub repo): 106
New variety images: 100
Total: 206


In [16]:
import csv
import os

# Combine both variety batches info
all_variety_files = sorted(os.listdir('/content/drive/MyDrive/data/raw/variety'))

print(f'Found {len(all_variety_files)} variety images to add to labels.csv')
for f in all_variety_files[:5]:
    print(f)

Found 100 variety images to add to labels.csv
variety2_10_nastaliq_48.png
variety2_11_nastaliq_48.png
variety2_12_nastaliq_36.png
variety2_13_nastaliq_48.png
variety2_14_naskh_48.png


In [17]:
import os
import csv
import re

# Re-list the same text batches used earlier (must match exactly)
batch1_texts = [
    "پاکستان کا دارالحکومت اسلام آباد ہے","چاند رات کو بہت خوبصورت لگتا ہے","ہر انسان کو محنت کرنی چاہیے",
    "کتابیں علم کا خزانہ ہیں","بارش کے بعد ہوا تازہ ہو جاتی ہے","طالب علم کو وقت کی قدر کرنی چاہیے",
    "دوستی میں اعتماد بہت ضروری ہے","صبح جلدی اٹھنا صحت کے لیے اچھا ہے","ماں باپ کی خدمت بہت بڑی نیکی ہے",
    "ایمانداری بہترین پالیسی ہے","غریبوں کی مدد کرنی چاہیے","علم حاصل کرنے کی کوئی عمر نہیں",
    "پھول خوشبو اور خوبصورتی کی علامت ہیں","پرندے آزادی کی علامت ہیں","زندگی میں صبر کا پھل میٹھا ہوتا ہے",
    "نیک اعمال انسان کو کامیاب بناتے ہیں","تعلیم ہر انسان کا بنیادی حق ہے","محنتی انسان کبھی ناکام نہیں ہوتا",
    "وقت پر کام کرنا کامیابی کی کنجی ہے","دوسروں کی عزت کرنی چاہیے"
]

batch2_texts = [
    "علم بغیر عمل کے بیکار ہے","سچ بولنا ہمیشہ فائدہ مند ہوتا ہے","ہمسائیوں کے ساتھ اچھا سلوک کریں",
    "درخت لگانا ماحول کے لیے مفید ہے","پانی زندگی کے لیے بہت ضروری ہے","بچوں کی تربیت والدین کی ذمہ داری ہے",
    "کامیابی کے لیے مستقل مزاجی ضروری ہے","اچھی عادتیں انسان کو بہتر بناتی ہیں","دنیا میں امن بہت ضروری ہے",
    "صحت مند رہنے کے لیے ورزش کریں","کتاب انسان کی بہترین دوست ہے","وقت کبھی واپس نہیں آتا",
    "نیکی کبھی ضائع نہیں ہوتی","علم کی روشنی سے جہالت دور ہوتی ہے","محنت کے بغیر کامیابی ممکن نہیں",
    "غصہ انسان کی سب سے بڑی کمزوری ہے","دوسروں کی مدد کرنا نیکی ہے","استاد قوم کا معمار ہوتا ہے",
    "زندگی مشکلات کا نام ہے","امید پر دنیا قائم ہے","خواب دیکھنے سے کچھ نہیں ہوتا، عمل کرنا ضروری ہے",
    "سچی محنت رنگ لاتی ہے","علم کے بغیر انسان ادھورا ہے","قوم کی ترقی تعلیم میں ہے",
    "اچھے دوست زندگی کا سرمایہ ہیں","ایمانداری کامیابی کی کنجی ہے","بڑوں کا احترام کرنا چاہیے",
    "غریبی میں بھی خوداری ضروری ہے","زندگی میں صبر کا پھل میٹھا ہے","علم انسان کو باشعور بناتا ہے",
    "دیانتداری بہترین خوبی ہے","قناعت سب سے بڑی دولت ہے","دوسروں پر اعتماد کرنا سیکھیں",
    "ہر کام میں برکت نیت پر ہے","زندگی ایک سفر ہے، منزل نہیں","علم عمل کے بغیر بے سود ہے",
    "قوم کی تعمیر نوجوانوں کے ہاتھ میں ہے","سچائی کا راستہ مشکل مگر روشن ہے","کامیابی کے پیچھے قربانی ہوتی ہے",
    "زندگی میں مقصد رکھنا ضروری ہے"
]

batch3_texts = [
    "اچھے اخلاق انسان کی پہچان ہیں","علم کے بغیر ترقی ممکن نہیں","محنت کبھی رائیگاں نہیں جاتی",
    "زندگی میں ہمت نہیں ہارنی چاہیے","دوسروں کے کام آنا بڑی نیکی ہے","وقت کی پابندی کامیابی کی نشانی ہے",
    "بچوں کو اچھی تربیت دینی چاہیے","صبر اور استقامت سے سب کچھ ممکن ہے","غرور انسان کو تباہ کر دیتا ہے",
    "علم روشنی ہے اور جہالت اندھیرا","اپنے وطن سے محبت کرنی چاہیے","سچائی اور ایمانداری کامیابی کی ضمانت ہے",
    "زندگی میں نظم و ضبط ضروری ہے","استاد کا احترام کرنا فرض ہے","علم انسان کی معراج ہے",
    "قوموں کی ترقی محنت سے ہوتی ہے","دوسروں کے جذبات کا خیال رکھیں","کامیاب لوگ ہمیشہ محنتی ہوتے ہیں",
    "زندگی میں مثبت سوچ رکھیں","دیانتداری سب سے بڑی خوبی ہے","علم حاصل کرنا فرض ہے",
    "دوستی میں خلوص ضروری ہے","ماحول کی حفاظت ہماری ذمہ داری ہے","بڑوں کی بات سننی چاہیے",
    "زندگی میں شکر گزار رہنا چاہیے","اچھے کام کا صلہ ضرور ملتا ہے","علم سے انسان کی سوچ بدلتی ہے",
    "دوسروں کی مدد سے دل کو سکون ملتا ہے","زندگی میں توازن رکھنا ضروری ہے","سچی خوشی دوسروں کو خوش دیکھ کر ملتی ہے",
    "علم کی تلاش کبھی ختم نہیں ہوتی","قوم کی خدمت سب سے بڑی عبادت ہے","اچھی صحبت انسان کو بہتر بناتی ہے",
    "زندگی میں ایمانداری کا دامن نہ چھوڑیں","علم و عمل کا امتزاج کامیابی ہے","دوسروں کی عزت کرنے سے عزت ملتی ہے",
    "زندگی میں مقصد کا تعین ضروری ہے","سچی محنت ہمیشہ رنگ لاتی ہے","علم انسان کو باوقار بناتا ہے",
    "قناعت پسندی زندگی کو آسان بناتی ہے"
]

variety_folder = '/content/drive/MyDrive/data/raw/variety'
all_variety_files = sorted(os.listdir(variety_folder))

new_rows = []
for filename in all_variety_files:
    match = re.match(r'variety(\d*)_(\d+)_(\w+)_(\d+)\.png', filename)
    if match:
        batch_num = match.group(1)  # '' for batch1, '2' for batch2, '3' for batch3
        idx = int(match.group(2)) - 1
        font_name = match.group(3)
        size = match.group(4)

        if batch_num == '':
            text = batch1_texts[idx]
        elif batch_num == '2':
            text = batch2_texts[idx]
        elif batch_num == '3':
            text = batch3_texts[idx]
        else:
            text = 'UNKNOWN'

        new_rows.append({
            'filename': filename,
            'text': text,
            'category': 'variety',
            'font': font_name,
            'size': size
        })

print(f'Matched {len(new_rows)} out of {len(all_variety_files)} files')
print(new_rows[:3])  # preview first 3

Matched 100 out of 100 files
[{'filename': 'variety2_10_nastaliq_48.png', 'text': 'صحت مند رہنے کے لیے ورزش کریں', 'category': 'variety', 'font': 'nastaliq', 'size': '48'}, {'filename': 'variety2_11_nastaliq_48.png', 'text': 'کتاب انسان کی بہترین دوست ہے', 'category': 'variety', 'font': 'nastaliq', 'size': '48'}, {'filename': 'variety2_12_nastaliq_36.png', 'text': 'وقت کبھی واپس نہیں آتا', 'category': 'variety', 'font': 'nastaliq', 'size': '36'}]


In [18]:
import csv

with open('repo_data/data/labels.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    first_few_rows = [next(reader) for _ in range(3)]

print('Header:', header)
print('Sample rows:')
for row in first_few_rows:
    print(row)

Header: ['image', 'text']
Sample rows:
['data/synthetic/urdu_1.png', 'اردو ایک خوبصورت زبان ہے']
['data/synthetic/urdu_2.png', 'پاکستان ہمارا وطن ہے']
['data/synthetic/urdu_3.png', 'یہ ایک مصنوعی ذہانت کا منصوبہ ہے']


In [19]:
import csv
import os
import re

variety_folder = '/content/drive/MyDrive/data/raw/variety'
all_variety_files = sorted(os.listdir(variety_folder))

new_rows = []
for filename in all_variety_files:
    match = re.match(r'variety(\d*)_(\d+)_(\w+)_(\d+)\.png', filename)
    if match:
        batch_num = match.group(1)
        idx = int(match.group(2)) - 1

        if batch_num == '':
            text = batch1_texts[idx]
        elif batch_num == '2':
            text = batch2_texts[idx]
        elif batch_num == '3':
            text = batch3_texts[idx]
        else:
            text = 'UNKNOWN'

        # Match the same path format as existing labels.csv
        image_path = f'data/raw/variety/{filename}'
        new_rows.append([image_path, text])

print(f'Prepared {len(new_rows)} new rows')
print('Preview:')
for row in new_rows[:3]:
    print(row)

Prepared 100 new rows
Preview:
['data/raw/variety/variety2_10_nastaliq_48.png', 'صحت مند رہنے کے لیے ورزش کریں']
['data/raw/variety/variety2_11_nastaliq_48.png', 'کتاب انسان کی بہترین دوست ہے']
['data/raw/variety/variety2_12_nastaliq_36.png', 'وقت کبھی واپس نہیں آتا']


In [20]:
import csv

# Read existing labels.csv
with open('repo_data/data/labels.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    existing_rows = list(reader)

print(f'Existing rows: {len(existing_rows)}')
print(f'New rows to add: {len(new_rows)}')

# Combine existing + new rows
all_rows = existing_rows + new_rows

# Write the updated labels.csv (locally first, in this Colab session)
with open('labels_updated.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(all_rows)

print(f'Total rows in updated file: {len(all_rows)}')
print('Saved as labels_updated.csv')

Existing rows: 69
New rows to add: 100
Total rows in updated file: 169
Saved as labels_updated.csv


In [21]:
from google.colab import files
files.download('labels_updated.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
!pip install transformers torch pillow pandas

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor, base_folder=''):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.base_folder = base_folder
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = f"{self.base_folder}/{row['image']}" if self.base_folder else row['image']
        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

In [27]:
# Load the TrOCR processor
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

# Create dataset (with base_folder pointing to the cloned repo)
dataset = UrduOCRDataset('repo_data/data/labels.csv', processor, base_folder='repo_data')

# Test it loads correctly
sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 69 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 55
Testing samples: 14


In [28]:
from google.colab import files
files.download('labels_updated.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
import os
variety_files = os.listdir('/content/drive/MyDrive/data/raw/variety')
print(f'Total files in variety folder: {len(variety_files)}')
print(variety_files[:10])  # show first 10 filenames

Total files in variety folder: 100
['variety_1_naskh_24.png', 'variety_2_naskh_24.png', 'variety_3_naskh_24.png', 'variety_4_nastaliq_48.png', 'variety_5_nastaliq_24.png', 'variety_6_nastaliq_36.png', 'variety_7_naskh_48.png', 'variety_8_naskh_24.png', 'variety_9_nastaliq_48.png', 'variety_10_naskh_48.png']
